# 05: Structure Module

The structure module turns Evoformer features into one rigid frame per residue. Each frame stores a position and orientation; its origin becomes the predicted C-alpha coordinate in this compact model.

The result is a 3D C-alpha backbone trace, not a complete atomic structure. Side-chain torsions and all-atom reconstruction are intentionally omitted.

**Read alongside:** `../src/af2_from_scratch/structure_module.py`

## Stage map

```text
single features s + pair features z + current residue frames T
                              |
                              v
                 invariant point attention
                              |
                              v
                    update single features
                              |
                              v
             predict local rotation and translation
                              |
                              v
                    compose with current frames
                              |
                         repeat n_ipa times
                              |
                              v
                 residue frames + C-alpha trace
```

All iterations reuse the same attention, transition, and backbone-update parameters. Increasing `n_ipa` therefore adds computation but not parameters.

In [ ]:
import sys
from dataclasses import replace

import torch

sys.path.insert(0, "../src")
torch.manual_seed(0)

## 1. Attend with features and 3D points

Invariant point attention (IPA) extends ordinary attention with geometry. For each residue and attention head, it predicts scalar query, key, and value features as well as query, key, and value **points**.

The points begin in each residue's local frame. IPA moves them into global coordinates and combines three score terms:

```text
attention score = content match + pair bias - weighted squared point distance
```

Nearby query and key points receive a smaller distance penalty. Learned positive weights control how strongly each head uses geometry.

After attention, weighted value points are moved back into the query residue's local frame. If every input frame is rotated and translated together, the IPA feature update stays the same. This is the precise meaning of **invariant** here.

In [ ]:
from af2_from_scratch import AF2Config
from af2_from_scratch.geometry import make_T, quat_to_rot
from af2_from_scratch.structure_module import IPA

cfg = AF2Config()
n_res = 12
s = torch.randn(n_res, cfg.c_s)
z = torch.randn(n_res, n_res, cfg.c_z)
frames = make_T(torch.eye(3).expand(n_res, 3, 3), torch.randn(n_res, 3))
ipa = IPA(cfg).eval()

with torch.no_grad():
    original_update = ipa(s, z, frames)
    global_transform = make_T(
        quat_to_rot(torch.tensor([1.0, 0.2, -0.1, 0.3])),
        torch.tensor([2.0, -1.0, 3.0]),
    )
    moved_update = ipa(s, z, global_transform @ frames)

print("IPA input/output:", tuple(s.shape), "->", tuple(original_update.shape))
print(
    "largest change after global rigid motion:",
    (original_update - moved_update).abs().max().item(),
)

## 2. Refine residue frames iteratively

The structure module starts every residue with an identity frame at the origin. Each iteration then:

1. Updates `s` with IPA using the current frames.
2. Applies a transition network to `s`.
3. Predicts a six-number backbone update: three values for a quaternion and three for a translation. This compact implementation scales the translation by 0.1 for stability.
4. Composes that local update with the current frame.

The quaternion's real component is fixed to one before normalization. Between iterations, rotation gradients are stopped as in AlphaFold, while translation gradients continue. This stabilizes optimization without changing the forward geometry.

In [ ]:
from af2_from_scratch.structure_module import StructureModule

structure_module = StructureModule(cfg).eval()
with torch.no_grad():
    output_frames, refined_s = structure_module(s, z)
ca_trace = output_frames[..., :3, 3]

more_iterations = StructureModule(replace(cfg, n_ipa=cfg.n_ipa + 1))
parameter_count = sum(p.numel() for p in structure_module.parameters())
more_iteration_count = sum(p.numel() for p in more_iterations.parameters())

print("frames:", tuple(output_frames.shape))
print("C-alpha trace:", tuple(ca_trace.shape))
print("refined single features:", tuple(refined_s.shape))
print(f"structure-module parameters: {parameter_count / 1e3:.0f}k")
print("extra iteration adds parameters:", more_iteration_count != parameter_count)

## 3. Understand the untrained output

The final backbone-update layer is initialized to zero. Before training, every predicted update is therefore the identity transform and every C-alpha origin remains at zero.

This collapsed output is deliberate. It gives optimization a stable starting point; it should not be interpreted as an initial structure prediction. Training changes the update layer, after which different residue features produce different rotations and translations.

In [ ]:
rotations = output_frames[..., :3, :3]
identity = torch.eye(3).expand_as(rotations)

print("largest initial C-alpha displacement:", ca_trace.abs().max().item())
print(
    "largest initial rotation error:",
    (rotations.transpose(-1, -2) @ rotations - identity).abs().max().item(),
)

**Recap:** IPA combines learned sequence features, pair features, and distances between learned 3D points. Repeated local frame updates then produce one position and orientation per residue.

Next, `06_full_model.ipynb` connects feature extraction, embedding, the Evoformer, recycling, and the structure module.